In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/DARPA_OPTC_EDA')
REPO_URL = 'https://github.com/farzaandalwai/DARPA_OPTC_EDA.git'
BRANCH = 'eda08'
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    if not (REPO / '.git').exists() or git('remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('STOP: existing code directory is not the expected repository')
    if git('status', '--porcelain'):
        raise RuntimeError('STOP: code checkout has local changes; preserve them before syncing')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)


In [ ]:
CHECKED_OUT_BRANCH = git('branch', '--show-current')
CODE_HEAD = git('rev-parse', 'HEAD')
print('Checked-out branch:', CHECKED_OUT_BRANCH)
print('Exact git HEAD:', CODE_HEAD)
assert CHECKED_OUT_BRANCH == BRANCH


In [ ]:
import os
DATA_ROOT = Path('/content/drive/MyDrive/DARPA_OPTC_EDA')
assert os.path.ismount('/content/drive'), 'STOP: Google Drive is not mounted'
assert Path('/content/drive/MyDrive').is_dir(), 'STOP: MyDrive is unavailable'
assert DATA_ROOT.is_dir(), 'STOP: DARPA_OPTC_EDA artifact folder is unavailable'
print('Google Drive mounted; artifact root:', DATA_ROOT)


# SysClient0201: first real hybrid connectivity run

This notebook only invokes repository logic. No model training, feature fitting, high-degree pruning, or EDA10 changes. Code stays in /content; original Drive artifacts are read-only.

The 2026-10-01 real-artifact audit found that period_process_activity_raw_v1.parquet is explicitly period-assigned but contains only per-process aggregates, not behavior action/object/context keys. With these known inputs the audit will STOP before either graph is constructed. Do not bypass this gate or substitute the original full-timeline compact behavior table. A different behavior artifact requires inspection and explicit period assignments.

The established policy selects exactly 70 PROCESS features. reference_full_structure_id supplies the retained full EDA10 CREATE-family structure_id; split_structure_id remains separate metadata. Each graph has period-isolated context identity.


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)


In [ ]:
import json
import tempfile

WORK = Path(tempfile.mkdtemp(prefix='sysclient0201_hybrid_', dir='/content'))
RUNNER = REPO / 'src/eda/prepare_sysclient0201_hybrid_graph.py'
assert RUNNER.is_file(), 'STOP: current eda08 is missing the repository runner'
print('Local audit and temporary behavior inputs:', WORK)
print('Code version for this run:', CODE_HEAD)

# Set ONLY after inspecting a different, explicitly period-assigned behavior artifact.
# Never point this at full-timeline compact behavior.
BEHAVIOR_TABLE_OVERRIDE = None
ARGS = ['--data-root', str(DATA_ROOT), '--work-dir', str(WORK)]
if BEHAVIOR_TABLE_OVERRIDE is not None:
    ARGS += ['--behavior-table', str(BEHAVIOR_TABLE_OVERRIDE)]


In [ ]:
# Prints existence, hashes, row counts, columns/types, roles, mappings, exact
# 70 features, CREATE reconciliation, and any hard blocker. No graph writes.
audit_result = subprocess.run([sys.executable, str(RUNNER), 'audit', *ARGS], cwd=REPO)
audit_report = json.loads((WORK / 'compatibility_audit.json').read_text())
print('Final 70-feature policy passed:', audit_report['final_70_feature_policy_passed'])
print('Real-artifact compatibility audit passed:', audit_report['audit_passed'])
if audit_result.returncode != 0 or not audit_report['audit_passed']:
    for error in audit_report['errors']:
        print('BLOCKER:', error)
    print('Neither graph was built by this run. Intended outputs:', audit_report['output_paths'])
    raise RuntimeError('STOP: incompatible real artifacts; obtain period-assigned behavior links')


In [ ]:
# Re-audits; partitions ONLY by explicit period_role, never time ranges.
# This guard remains effective even if a failed audit cell is skipped.
subprocess.run([sys.executable, str(RUNNER), 'prepare', *ARGS], cwd=REPO, check=True)


In [ ]:
# Verifies source/prepared hashes and refuses either existing output directory.
# Calls period_heterogeneous_graph_v1 once per period; no shared graph state.
subprocess.run([sys.executable, str(RUNNER), 'build', *ARGS], cwd=REPO, check=True)


In [ ]:
# PROCESS/context counts, forward/CREATE edges, WCCs, largest PROCESS share,
# shared context/families, max/p95 degree and top 10; old summary if readable.
subprocess.run([sys.executable, str(RUNNER), 'compare', *ARGS], cwd=REPO, check=True)
status = json.loads((WORK / 'execution_status.json').read_text())
print('Commit SHA:', CODE_HEAD)
print('Audit passed:', json.loads((WORK / 'compatibility_audit.json').read_text())['audit_passed'])
print('Completed graphs:', status['completed_periods'])
print('Output Drive paths:', audit_report['output_paths'])
print('Errors:', status['errors'])
